# Model Internal Consistency

Check how consistent the model's internal representations are:
logit lens agreement, norm monotonicity, component orthogonality,
and output sensitivity.

## Why This Matters

Model internal consistency provides tools for systematic analysis and comparison of transformer internals. These diagnostics help identify unexpected behaviors, compare architectures, and build a comprehensive picture of how the model processes information.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.model_internal_consistency import (
    logit_lens_consistency, residual_norm_monotonicity,
    component_orthogonality, output_sensitivity,
    model_consistency_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Logit Lens Consistency

How much does each layer change the predicted token?

In [ ]:
result = logit_lens_consistency(model, tokens, position=-1)
print(f"Total changes: {result['n_changes']}")
print(f"Final entropy: {result['final_entropy']:.4f}")
for p in result['per_layer']:
    flag = ' [CHANGE]' if p['changed'] else ''
    print(f"  Layer {p['layer']}: pred={p['predicted_token']} "
          f"prob={p['max_prob']:.4f}{flag}")

## Residual Norm Monotonicity

Does the residual stream norm grow consistently through the model?

In [ ]:
result = residual_norm_monotonicity(model, tokens)
print(f"Monotonic: {result['is_monotonic']}")
print(f"Violations: {result['n_violations']}")
for p in result['per_layer']:
    bar = '█' * int(p['mean_norm'] * 5)
    flag = ' [VIOLATION]' if p.get('violation', False) else ''
    print(f"  Layer {p['layer']}: norm={p['mean_norm']:.4f} {bar}{flag}")

## Component Orthogonality

How orthogonal are attention and MLP outputs at each layer?

In [ ]:
result = component_orthogonality(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: cos(attn,mlp)={p['mean_cosine']:.4f} "
          f"{'ORTHOGONAL' if abs(p['mean_cosine']) < 0.3 else 'ALIGNED'}")

## Output Sensitivity

How much does the output change per unit of input change?

In [ ]:
result = output_sensitivity(model, tokens, n_perturbations=5)
print(f"Mean sensitivity: {result['mean_sensitivity']:.4f}")
print(f"Max sensitivity: {result['max_sensitivity']:.4f}")
for p in result['per_perturbation']:
    bar = '█' * int(p['sensitivity'] * 2)
    print(f"  Trial {p['trial']}: sensitivity={p['sensitivity']:.4f} {bar}")

## Consistency Summary

In [ ]:
result = model_consistency_summary(model, tokens)
print(f"Prediction changes: {result['n_prediction_changes']}")
print(f"Norm monotonic: {result['norm_monotonic']}")
print(f"Mean orthogonality: {result['mean_orthogonality']:.4f}")
print(f"Mean sensitivity: {result['mean_sensitivity']:.4f}")